This code does not manage to find a good performing neural network, due to computational constraints. The hyperparameter grid contains only 81 combinations. More values can easily be added In order to perform more extensive hyperparameter tuning, resulting in better performing models. The layer configuration is fixed to [32, 16, 8], which was found to perform good by Gu, Kelly and Xiu (2020). 

The Tensorflow library is not compatible with python 3.13. This code has executed using Python 3.12.10.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from itertools import product
import gc

# Ensure reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Define validation length
validation_length = 12

# Updated data split
estimation_periods = pd.DataFrame({
    'oos_year': range(1987, 2017),
    'validation_end': range(1986, 2016),
    'validation_start': range(1975, 2005),
    'training_start': [1957] * 30,
    'training_end': range(1974, 2004)
})[['training_start', 'training_end', 'validation_start', 'validation_end', 'oos_year']]

print("Estimation Periods:")
print(estimation_periods.head())
print(estimation_periods.tail())

# Generate visualization data
visualization_data = []
for _, row in estimation_periods.iterrows():
    for year in range(row['training_start'], row['oos_year'] + 1):
        if row['training_start'] <= year <= row['training_end']:
            classification = 'Training'
        elif row['validation_start'] <= year <= row['validation_end']:
            classification = 'Validation'
        elif year == row['oos_year']:
            classification = 'OOS'
        else:
            continue
        visualization_data.append({
            'year': year,
            'oos_year': row['oos_year'],
            'classification': classification
        })

visualization_data = pd.DataFrame(visualization_data)

# Visualize timeline
plt.figure(figsize=(10, 6))
sns.scatterplot(data=visualization_data, x='year', y='oos_year', hue='classification', size=1, legend=True)
plt.title("Data Classification Timeline")
plt.xlabel(None)
plt.ylabel(None)
plt.yticks([])
plt.savefig('timeline_tuned_dropout_adam.png')
plt.close()

def load_year_data(years, data_path= 'YOUR_DATA_PATH'):  # Change this to your data path
    if data_path == 'YOUR_DATA_PATH':
        raise ValueError("Please provide a valid data_path")
    data = []
    for year in years:
        file = f"{data_path}/year_{year}.parquet"
        if os.path.exists(file):
            data.append(pd.read_parquet(file))
    return pd.concat(data, ignore_index=True) if data else pd.DataFrame()

def build_model(input_dim, layer_config, l1_reg, dropout_rate, learning_rate):
    model = Sequential()
    model.add(Dense(layer_config[0], activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l1(l1_reg)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    for units in layer_config[1:]:
        model.add(Dense(units, activation='relu', kernel_regularizer=regularizers.l1(l1_reg)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model


#---------------------------------------------------------#
# HERE YOU CAN CHANGE THE HYPERPARAMETERS OR ADD NEW ONES
#---------------------------------------------------------#

# Define hyperparameter grid
param_grid = {
    'l1_reg': [1e-5, 1e-4, 1e-3],
    'dropout_rate': [0.1, 0.2, 0.3],
    'learning_rate': [1e-3, 3e-3, 1e-2],
    'batch_size': [128, 256, 512]
}
layer_config = [32, 16, 8]  # Fixed architecture

n_ensemble = 3

monthly_portfolio_returns = []
results = []

for _, row in tqdm(estimation_periods.iterrows(), total=len(estimation_periods), desc="Processing years"):
    oos_year = row['oos_year']
    pred_file = f"predictions_tuned_dropout_adam/pred_year_{oos_year}.parquet"
    
    if os.path.exists(pred_file):
        oos_data = pd.read_parquet(pred_file)
    else:
        train_years = range(row['training_start'], row['training_end'] + 1)
        val_years = range(row['validation_start'], row['validation_end'] + 1)
        oos_years = [oos_year]
        
        train_data = load_year_data(train_years)
        val_data = load_year_data(val_years)
        oos_data = load_year_data(oos_years)
        
        if train_data.empty or val_data.empty or oos_data.empty:
            print(f"Skipping OOS year {oos_year}: Missing data")
            continue
        
        # Standardize month format
        for df in [train_data, val_data, oos_data]:
            if 'month' in df.columns:
                df['month'] = pd.to_datetime(df['month'], format='%Y-%m')
        
        # Select all feature columns
        exclude = ['permno', 'month', 'mktcap_lag', 'ret_excess', 'year']
        exclude = [c for c in exclude if c in train_data.columns]
        feature_cols = [col for col in train_data.columns if col not in exclude]
        
        X_train = train_data[feature_cols].values
        X_val = val_data[feature_cols].values
        X_oos = oos_data[feature_cols].values
        
        y_train = train_data['ret_excess'].values
        y_val = val_data['ret_excess'].values
        y_oos = oos_data['ret_excess'].values
        
        best_val_loss = float('inf')
        best_params = None
        best_y_pred_oos = np.zeros_like(y_oos, dtype=float)
        
        # Iterate through all hyperparameter combinations
        for l1_reg, dropout_rate, learning_rate, batch_size in product(
            param_grid['l1_reg'],
            param_grid['dropout_rate'],
            param_grid['learning_rate'],
            param_grid['batch_size']
        ):
            ensemble_predictions_val = np.zeros_like(y_val, dtype=float)
            ensemble_predictions_oos = np.zeros_like(y_oos, dtype=float)
            for seed in range(n_ensemble):
                try:
                    tf.random.set_seed(42 + seed)
                    np.random.seed(42 + seed)
                    model = build_model(
                        input_dim=X_train.shape[1],
                        layer_config=layer_config,
                        l1_reg=l1_reg,
                        dropout_rate=dropout_rate,
                        learning_rate=learning_rate
                    )
                    early_stopping = EarlyStopping(
                        monitor='val_loss',
                        patience=5,
                        restore_best_weights=True,
                        min_delta=0.0
                    )
                    model.fit(
                        X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=50,
                        batch_size=batch_size,
                        callbacks=[early_stopping],
                        verbose=1
                    )
                    ensemble_predictions_val += model.predict(X_val, verbose=0).flatten() / n_ensemble
                    ensemble_predictions_oos += model.predict(X_oos, verbose=0).flatten() / n_ensemble
                    del model
                    tf.keras.backend.clear_session()
                    gc.collect()
                except Exception as e:
                    print(f"Error training model for params {l1_reg, dropout_rate, learning_rate, batch_size}: {e}")
                    continue
            
            # Average ensemble predictions for validation
            val_loss = np.mean((y_val - ensemble_predictions_val) ** 2)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params = {
                    'layer_config': layer_config,
                    'l1_reg': l1_reg,
                    'dropout_rate': dropout_rate,
                    'learning_rate': learning_rate,
                    'batch_size': batch_size
                }
                best_y_pred_oos = ensemble_predictions_oos
        
        print(f"OOS Year {oos_year}: Best params: {best_params}, Val Loss: {best_val_loss:.4f}")
        
        oos_data['y_pred'] = best_y_pred_oos
        os.makedirs("predictions_tuned_dropout_adam", exist_ok=True)
        oos_data[['permno', 'month', 'mktcap_lag', 'ret_excess', 'y_pred']].to_parquet(pred_file)
    
    # Form portfolios
    for month, month_data in oos_data.groupby('month'):
        month_data = month_data.copy()
        try:
            month_data['decile'] = pd.qcut(month_data['y_pred'], 10, labels=False, duplicates='drop') + 1
            num_bins = month_data['decile'].nunique()
            print(f"Month {month}: {num_bins} bins created")
            ew_returns = month_data.groupby('decile')['ret_excess'].mean()
            vw_returns = month_data.groupby('decile').apply(
                lambda x: (x['ret_excess'] * x['mktcap_lag']).sum() / x['mktcap_lag'].sum() if x['mktcap_lag'].sum() != 0 else 0
            )
            if 1 in ew_returns.index and max(ew_returns.index) in ew_returns.index:
                ew_portfolio_return = ew_returns[max(ew_returns.index)] - ew_returns[1]
                vw_portfolio_return = vw_returns[max(vw_returns.index)] - vw_returns[1]
                monthly_portfolio_returns.append({
                    'month': month,
                    'ew_portfolio_return': ew_portfolio_return,
                    'vw_portfolio_return': vw_portfolio_return
                })
            else:
                print(f"Skipping month {month}: Missing decile 1 or max decile")
        except ValueError as e:
            print(f"Skipping month {month}: {e}")
    
    # Calculate R-squared
    ss_res = np.sum((oos_data['ret_excess'] - oos_data['y_pred']) ** 2)
    ss_tot = np.sum((oos_data['ret_excess'] - oos_data['ret_excess'].mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0
    
    # Calculate Naive R-squared
    ss_res = np.sum((oos_data['ret_excess'] - oos_data['y_pred']) ** 2)
    ss_tot_naive = np.sum((oos_data['ret_excess'] - 0) ** 2)
    naive_r2 = 1 - ss_res / ss_tot_naive if ss_tot_naive != 0 else 0
    
    results.append({
        'oos_year': oos_year,
        'mae': np.mean(np.abs(oos_data['ret_excess'] - oos_data['y_pred'])),
        'mse': np.mean((oos_data['ret_excess'] - oos_data['y_pred']) ** 2),
        'r2': r2,
        'naive_r2': naive_r2,
        'num_train': len(train_data) if 'train_data' in locals() else 0,
        'num_val': len(val_data) if 'val_data' in locals() else 0,
        'num_oos': len(oos_data),
        'best_params': best_params,
        'best_val_loss': best_val_loss
    })
    
    # Save results incrementally
    results_df = pd.DataFrame(results)
    results_df.to_csv("nn_tuned_dropout_adam_results.csv", index=False)
    
    del oos_data
    tf.keras.backend.clear_session()
    gc.collect()

# Save final portfolio returns
portfolio_returns = pd.DataFrame(monthly_portfolio_returns)
portfolio_returns['month'] = pd.to_datetime(portfolio_returns['month'], format='%Y-%m')
portfolio_returns = portfolio_returns.sort_values('month')

ew_mean = portfolio_returns["ew_portfolio_return"].mean()
ew_std = portfolio_returns['ew_portfolio_return'].std()
ew_sharpe = (ew_mean / ew_std) * np.sqrt(12) if ew_std != 0 else np.nan

vw_mean = portfolio_returns['vw_portfolio_return'].mean()
vw_std = portfolio_returns['vw_portfolio_return'].std()
vw_sharpe = (vw_mean / vw_std) * np.sqrt(12) if vw_std != 0 else np.nan

print(f"Equal-weighted Sharpe Ratio: {ew_sharpe:.2f}")
print(f"Value-weighted Sharpe Ratio: {vw_sharpe:.2f}")

portfolio_returns.to_csv("portfolio_tuned_dropout_adam_returns.csv", index=False)